# Test para armonización 

## 1. Configuración inicial ⚙️

Quería hacer una estructura más pro tipo proyecto, pero me daba problemas al cargar los .py de src en el notebook. Los meto en celdas y los oculto. Igualmente meto esto por si lo cambiamos más adelante.

In [23]:
# --- CONFIGURACIÓN GLOBAL ---

import sys, os, pandas as pd

# Asegurar que el notebook ve el proyecto raíz
sys.path.append(os.path.abspath(".."))

# Ajustes visuales
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 10)
pd.set_option("display.precision", 2)

print("Entorno preparado ✅")


Entorno preparado ✅


## 2. Configuración de datasets 

Esto iría en config.py ...

In [37]:
GENIUS_CONFIG = {
    "path": "../data/song_lyrics.csv",
    "usecols": ["artist", "title", "lyrics", "tag", "year", "language"], # ¡Añadimos 'language'!
    "rename_map": {"tag": "genre"},
    "source": "genius",
    
    # 🚨 NUEVA CONFIGURACIÓN DE FILTRADO
    "filters": { 
        "language_col": "language",
        "target_language": "en" 
    }
}

TCC_CONFIG = {
    "path": "../data/tcc_ceds_music.csv",
    "usecols": ["artist_name", "track_name", "lyrics", "genre", "release_date"],
    "rename_map": {
        "artist_name": "artist",
        "track_name": "title",
        "release_date": "year"
    },
    "source": "tcc",
    # 💡 TCC no necesita filtro porque está limpio, pero lo añadimos para escalabilidad si se necesitara:
    "filters": {} 
}

print("Configuraciones cargadas ✅")


Configuraciones cargadas ✅


## 3.  Clase DatasetHarmonizer

In [40]:
import pandas as pd
import re
from pathlib import Path
import langcodes # ⬅️ 

class DatasetHarmonizer:
    """
    Clase para cargar, muestrear y armonizar diferentes datasets
    de letras de canciones, estandarizando su formato y contenido.
    """
    def __init__(self, config, chunksize=None, sample_blocks=None):
        self.config = config
        self.chunksize = chunksize
        self.sample_blocks = sample_blocks
        self.df = None
        self.dataset_name = self.config.get("source", "dataset").upper()

    # --- MÉTODOS PRIVADOS DE CARGA (_load) ---

    def _load_full(self, path, usecols):
        """Carga el dataset completo sin muestreo."""
        print("\n Cargando dataset completo (sin muestreo)...")
        self.df = pd.read_csv(path, usecols=usecols)
        print(f" Dataset completo cargado con {len(self.df)} filas")

    def _load_sampled(self, path, usecols):
        """Carga el dataset usando muestreo (chunking)."""
        print(f"🔎 Muestreo → chunksize={self.chunksize:,}, sample_blocks={self.sample_blocks or 'all'}")

        chunks = pd.read_csv(path, usecols=usecols, chunksize=self.chunksize)
        parts = []
        for i, chunk in enumerate(chunks):
            print(f" Leyendo bloque {i+1}")
            parts.append(chunk)
            if self.sample_blocks and i + 1 >= self.sample_blocks:
                break
        self.df = pd.concat(parts, ignore_index=True)
        print(f" Muestra creada con {len(self.df)} filas")

    # --- MÉTODOS PRINCIPALES ---

    def load_data(self):
        """Orquesta la carga de datos (completa o muestreada)."""
        print(f"\n Cargando dataset: {self.dataset_name}")
        
        path = Path(self.config["path"])
        usecols = self.config["usecols"]

        if self.chunksize:
            self._load_sampled(path, usecols)
        else:
            self._load_full(path, usecols)

        return self

    # --- MÉTODOS PRIVADOS DE ARMONIZACIÓN (_harmonize) ---

    def _filter_language(self):
        """
        Filtra el DataFrame por idioma usando langcodes para estandarizar 
        los códigos antes de la comparación.
        """
        filters = self.config.get("filters", {})
        language_col = filters.get("language_col")
        target_lang_raw = filters.get("target_language") # 'en' o 'english'

        if language_col and target_lang_raw and language_col in self.df.columns:
            initial_count = len(self.df)
            
            # Normalizamos el código de idioma objetivo (ej. 'english' -> 'en')
            target_lang_code = langcodes.find(target_lang_raw).language
            
            # --- Proceso de Normalización del DataFrame ---
            
            print(f"[{self.dataset_name}] 🌐 Normalizando códigos de idioma a '{target_lang_code}'...")
            
            # 1. Aplicar limpieza básica y luego la normalización de langcodes
            
            def normalize_lang_code(code):
                """Convierte códigos de idioma brutos (ej. 'EN', 'eng') a un estándar ('en')."""
                try:
                    # Convierte a string, minúsculas y luego usa langcodes para encontrar el código base
                    cleaned_code = str(code).strip().lower()
                    return langcodes.find(cleaned_code).language
                except:
                    return None 

            self.df["normalized_lang"] = self.df[language_col].apply(normalize_lang_code)
            
            # 2. Aplicar el filtro con el código normalizado
            self.df = self.df[self.df["normalized_lang"] == target_lang_code].copy()
            
            # 3. Limpieza final: Eliminamos las columnas auxiliares
            self.df.drop(columns=[language_col, "normalized_lang"], inplace=True) 
            
            final_count = len(self.df)
            print(f"[{self.dataset_name}] 🌐 Filtrado por idioma completado: {initial_count} filas -> {final_count} filas (solo '{target_lang_code}')")
        
        # Si no hay configuración de filtros o la columna no existe, simplemente retorna.


    def _clean_genius_lyrics(self):
        """Limpia las lyrics de Genius (elimina anotaciones, saltos de línea).
        Asume que la columna se llama 'lyrics'."""
        lyrics_col = "lyrics" 

        if lyrics_col not in self.df.columns:
            print(f"[{self.dataset_name}] ⚠️ No se encontró la columna '{lyrics_col}' para limpiar. Saltando.")
            return

        print(f"[{self.dataset_name}] 🧼 Aplicando limpieza específica de lyrics en '{lyrics_col}'...")
        
        # 1. Eliminar anotaciones entre corchetes [Chorus: X & Y]
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\[.*?\]", "", regex=True)
        
        # 2. Reemplazar saltos de línea (\n) con un espacio
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\n", " ", regex=True)
        
        # 3. Eliminar espacios múltiples y limpiar extremos
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\s+", " ", regex=True).str.strip()

        print(f"[{self.dataset_name}] ✅ Limpieza de lyrics finalizada.")

    def _clean_genius_lyrics(self):
        """Limpia las lyrics de Genius (elimina anotaciones, saltos de línea)."""
        lyrics_col = "lyrics" # Asumimos que la columna se llama 'lyrics'

        if lyrics_col not in self.df.columns:
            print(f"[{self.dataset_name}] ⚠️ No se encontró la columna '{lyrics_col}' para limpiar. Saltando.")
            return

        print(f"[{self.dataset_name}]  Aplicando procesado específico de lyrics en '{lyrics_col}'...")
        
        # 1. Eliminar anotaciones entre corchetes [Chorus: X & Y]
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\[.*?\]", "", regex=True)
        
        # 2. Reemplazar saltos de línea (\n) con un espacio
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\n", " ", regex=True)
        
        # 3. Eliminar espacios múltiples y limpiar extremos
        self.df[lyrics_col] = self.df[lyrics_col].str.replace(r"\s+", " ", regex=True).str.strip()

        print(f"[{self.dataset_name}] ✅ Limpieza de lyrics finalizada.")

    def _general_normalization(self):
        """Aplica renombrado de columnas y normalización de género/año/fuente."""
        print(f"[{self.dataset_name}]  Normalización general (renombre, género, año, fuente)...")
        
        # 1. Renombrado
        self.df.rename(columns=self.config["rename_map"], inplace=True)
        
        # 2. Normalización de género (minúsculas y limpieza)
        if "genre" in self.df.columns:
            self.df["genre"] = self.df["genre"].astype(str).str.lower().str.strip()
            
        # 3. Normalización de año
        if "year" in self.df.columns:
            self.df["year"] = pd.to_datetime(self.df["year"], errors="coerce").dt.year
            
        # 4. Añadir columna de fuente
        self.df["source"] = self.config["source"]
        

    def harmonize(self):
        """Orquesta todos los pasos de armonización y limpieza."""
        print(f"\n Iniciando armonización para: {self.dataset_name}")
        
        # Paso 1: Filtrado de idioma
        self._filter_language()
        
        # Paso 2: Limpieza específica de lyrics de Genius
        if self.dataset_name == "GENIUS":
            self._clean_genius_lyrics()
    
        # Paso 3: Normalización general de columnas y tipos de datos
        self._general_normalization()
        
        return self

    def get_df(self):
        return self.df

## 4. Lectura y armonización

In [ ]:


# Genius con muestreo
genius_loader = DatasetHarmonizer(GENIUS_CONFIG, chunksize=50000, sample_blocks=4) # Leemos 200k filas
df_genius = genius_loader.load_data().harmonize().get_df()

# TCC completo
tcc_loader = DatasetHarmonizer(TCC_CONFIG)
df_tcc = tcc_loader.load_data().harmonize().get_df()

# Unión
df_all = pd.concat([df_genius, df_tcc], ignore_index=True)



 Cargando dataset: GENIUS
🔎 Muestreo → chunksize=50,000, sample_blocks=4
 Leyendo bloque 1
 Leyendo bloque 2
 Leyendo bloque 3
 Leyendo bloque 4
 Muestra creada con 200000 filas

 Iniciando armonización para: GENIUS
[GENIUS] 🌐 Normalizando códigos de idioma a 'enc'...
[GENIUS] 🌐 Filtrado por idioma completado: 200000 filas -> 175359 filas (solo 'enc')
[GENIUS]  Aplicando procesado específico de lyrics en 'lyrics'...


In [35]:
print(" Data set de Genius:")
df_genius.head()

 Data set de Genius:


,title,genre,artist,year,lyrics,source


In [32]:
print(" Data set de TCC:")
df_tcc.head()

 Data set de TCC:


,artist,title,year,genre,lyrics,source
0,mukesh,mohabbat bhi jhoothi,1970,pop,hold time feel break feel untrue convince speak voice tear try hold hurt try forgive okay play break string feel heart want feel tell real truth hurt lie worse anymore little turn dust play house ...,tcc
1,frankie laine,i believe,1970,pop,believe drop rain fall grow believe darkest night candle glow believe go astray come believe believe believe smallest prayer hear believe great hear word time hear bear baby touch leaf believe bel...,tcc
2,johnnie ray,cry,1970,pop,sweetheart send letter goodbye secret feel better wake dream think real false emotions feel heartaches hang long blue get bluer song remember cloudy hair,tcc
3,pérez prado,patricia,1970,pop,kiss lips want stroll charm mambo chacha meringue heaven arm japan brag geisha care long uncle eye starry sort gleam like million dollar dream come true everybody wish steal heart away guess try e...,tcc
4,giorgos papadopoulos,apopse eida oneiro,1970,pop,till darling till matter know till dream live apart know hearts till world free oceans cross mountains climb know gain loss pray loss time till dream till memory till hold till oceans cross mounta...,tcc


In [36]:
print("✅ Dataset combinado con", len(df_all), "filas")
df_all.head()


✅ Dataset combinado con 28372 filas


,title,genre,artist,year,lyrics,source
0,mohabbat bhi jhoothi,pop,mukesh,1970,hold time feel break feel untrue convince speak voice tear try hold hurt try forgive okay play break string feel heart want feel tell real truth hurt lie worse anymore little turn dust play house ...,tcc
1,i believe,pop,frankie laine,1970,believe drop rain fall grow believe darkest night candle glow believe go astray come believe believe believe smallest prayer hear believe great hear word time hear bear baby touch leaf believe bel...,tcc
2,cry,pop,johnnie ray,1970,sweetheart send letter goodbye secret feel better wake dream think real false emotions feel heartaches hang long blue get bluer song remember cloudy hair,tcc
3,patricia,pop,pérez prado,1970,kiss lips want stroll charm mambo chacha meringue heaven arm japan brag geisha care long uncle eye starry sort gleam like million dollar dream come true everybody wish steal heart away guess try e...,tcc
4,apopse eida oneiro,pop,giorgos papadopoulos,1970,till darling till matter know till dream live apart know hearts till world free oceans cross mountains climb know gain loss pray loss time till dream till memory till hold till oceans cross mounta...,tcc


## Citas: Enlace Chat

https://gemini.google.com/share/eebeb9dfa33b